In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Windows


Autosaving every 180 seconds


auto.py (22): IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [39]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = r'F:\bmi\andres\Cohort 7 x4\DON-014375\day0\calibration\Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")

memmap :  (90000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (311, 90000)
         self.Fneu (neuropile):  (311, 90000)
         self.iscell (Suite2p cell classifier output):  (660, 2)
              of which number of good cells:  (311,)
         self.spks (deconnvoved spikes):  (311, 90000)
         self.stat (footprints structure):  (311,)
         mean std over all cells :  83.0
# of footprints;  311
DONE...


In [40]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

# 


computing roi traces for SNR indexing: 100%|███████████████████████████████████████| 9000/9000 [01:11<00:00, 126.02it/s]


In [41]:
#########################################################
########### VISUALIZE CELLS BY SNR OR F0 ##################
#########################################################
#
bmi_c.scale=1.5
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

bmi_c.min_f0 = 500
# visualize traces
bmi_c.vmin=0
bmi_c.vmax=3000
bmi_c.max_n_cells = 50
bmi_c.visualize_traces_snr_order(bmi_c.std_map)


memmap :  (90000, 512, 512)


In [36]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [21,24]
bmi_c.ensemble2 = [35,36]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles2(bmi_c.std_map)

print ("DONE...")


all cells: [21 24 35 36]


100%|███████████████████████████████████████████████████████████████████████████| 90000/90000 [01:28<00:00, 1018.11it/s]


Contour:  [ 63 427]
cell ids:  [21 24 35 36]
DONE...


In [37]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10
bmi_c.post_reward_lockout_baseline_min = 3
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.33#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
bmi_c.find_reward_thresholds_high_realtime()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  3000 max # of random rewards (i.e. every 30sec)  100
 @0.33% reward rate:  33
 high guess:  3.343867296870633
updated rewards #:  1 3.343867296870633
updated rewards #:  1 3.3104286239019265
updated rewards #:  1 3.277324337662907
updated rewards #:  1 3.244551094286278
updated rewards #:  1 3.212105583343415
updated rewards #:  1 3.1799845275099807
updated rewards #:  1 3.1481846822348807
updated rewards #:  1 3.1167028354125317
updated rewards #:  1 3.0855358070584065
updated rewards #:  2 3.0546804489878223
updated rewards #:  2 3.024133644497944
updated rewards #:  2 2.9938923080529642
updated rewards #:  2 2.9639533849724344
updated rewards #:  2 2.93431385112271
updated rewards #:  2 2.9049707126114828
updated rewards #:  3 2.875921005485368
updated rewards #:  4 2.8471617954305146
updated rewards #:  4 2.8186901774762094
updated rewards #:  4 2.790503275701447
updated rewards #:  4 2.7625982429444327
updated rewards #:  4 2.734972260514988
updated rewards #:  4 

In [38]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
 couldn't save bmi_c.object .... TO FIX!
Done...
